# PySpark — Data Engineering Interview Practice (80 questions)

**8 categories × 10 questions**. Run the setup cell first. DataFrames are loaded from the local `data/` folder. Use **PySpark DataFrame API** unless a question asks otherwise. Every exercise has its own independent code cell. For practice, use notebook 02; for reference, use notebook 03.

Before opening this notebook, install the repository's pinned dependencies from `requirements.txt`, which applies `constraints.txt`, in the practice virtual environment. The tested baseline uses **JDK 17**. Follow the [root README](../README.md) for environment setup, Java configuration, and kernel selection. This is **local Spark**, not a Databricks workspace.


> **Realistic Dataset v2:** Shared across Python, Pandas, and PySpark. Existing columns retained. Synthetic data includes intentional dirty records and malformed JSON for interview exercises.


In [1]:
import os, sys
# Use the active kernel's Python for local Spark workers; restart the kernel
# if Spark was previously started with another interpreter.
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window, types as T
from IPython.display import display

# Run from the extracted project directory in VS Code/Jupyter.
BASE = Path.cwd()
if not (BASE / 'data').exists():
    raise FileNotFoundError('Open the notebook from the pyspark_de_practice directory (data/ required).')
spark = (SparkSession.builder.master('local[2]').config('spark.pyspark.python', sys.executable)
       .config('spark.pyspark.driver.python', sys.executable).appName('PySpark DE Interview Practice')
         .config('spark.sql.session.timeZone','UTC')
         .config('spark.sql.shuffle.partitions','8')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

csv_tables = ['customers','products','orders','order_items','transactions','pipeline_logs','cdc_changes']
for name in csv_tables:
    globals()[name] = (spark.read.option('header',True).option('inferSchema',True)
                       .csv(str(BASE / 'data' / (name+'.csv'))))
# Explicit nested JSON input (avoid treating nested properties as strings).
events = spark.read.json(str(BASE / 'data' / 'events.jsonl'))
# Corrupt records are preserved in a string column; explicit schema is important.
malformed_events = (spark.read.schema(T.StructType([
    T.StructField('event_id',T.LongType()),
    T.StructField('event_type',T.StringType()),
    T.StructField('_corrupt_record',T.StringType())
])).option('mode','PERMISSIVE').option('columnNameOfCorruptRecord','_corrupt_record')
.json(str(BASE / 'data' / 'malformed_events.jsonl')))
print('Spark version:',spark.version)
for name in csv_tables+['events']:
    print(f'{name}: {globals()[name].count()} rows')


## 1. DataFrame Fundamentals


### Q01 — Preview orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Create a preview of `orders` containing ten records, or all records if fewer than ten are available. Retain every original column and its schema. Any ten input record occurrences are acceptable; return all records when fewer than ten exist. Neither a particular subset nor a particular output order is required. Do not transfer the complete dataset to the driver.

**Requirements:**
- Select only input record occurrences: a duplicate row cannot appear more times than it does in `orders`.
- Preserve the full schema and keep the preview distributed; neither sorting nor collecting the complete input is required.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with exactly the smaller of ten and the input record count, retaining the `orders` schema. Any valid subset of input occurrences is acceptable, with no required subset or row order.

**Solution:**


In [2]:
result = orders.limit(10)
result.show(20, truncate=False)


### Q02 — Project columns

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Create a DataFrame from `orders` containing only `order_id`, `customer_id`, and `amount`, in that column order. Retain every input record without changing the values or schema of the selected columns.

**Requirements:**
- Keep duplicate records and null values in the selected fields.
- Perform the projection on the distributed input without changing selected column types.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with exactly `order_id`, `customer_id`, and `amount`, in that order. Its record count equals the input count, and selected values and field schemas are unchanged.

**Solution:**


In [3]:
result = orders.select('order_id','customer_id','amount')
result.show(20, truncate=False)


### Q03 — Completed orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return complete records from `orders` whose `status` is exactly `completed`. Retain every original column for qualifying orders and leave the source DataFrame unchanged. Orders in any other status should be excluded.

**Requirements:**
- Match the stored status exactly; null statuses do not qualify.
- Filter the distributed orders without aggregating or deduplicating matching records.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame containing every completed-order occurrence and all original order columns. It retains the input schema; no matching orders yields an empty DataFrame with that schema.

**Solution:**


In [4]:
result = orders.filter(F.col('status') == 'completed')
result.show(20, truncate=False)


### Q04 — High value

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Filter `orders` to records whose `amount` is at least 300, including those exactly at the threshold. Return all original columns for the qualifying orders. Missing amounts do not qualify.

**Requirements:**
- Include 300 and exclude null amounts; do not normalize or round amounts before comparing.
- Retain all qualifying occurrences in the distributed result.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame of complete orders with `amount` at least 300. All original fields and their types remain, and no output sorting is required.

**Solution:**


In [5]:
result = orders.filter(F.col('amount') >= 300)
result.show(20, truncate=False)


### Q05 — Rename field

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return a version of `orders` with the `amount` column renamed to `order_amount`. Keep every record, retain all other columns, and preserve the renamed column's values and data type. Leave the original DataFrame unchanged.

**Requirements:**
- Rename rather than copy the field: the result should not also contain `amount`.
- Keep the rename as a distributed transformation without changing any field values.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `order_id`, `customer_id`, `order_date`, `status`, and `order_amount`. Record count and field types are unchanged; only the amount field's name differs.

**Solution:**


In [6]:
result = orders.withColumnRenamed('amount','order_amount')
result.show(20, truncate=False)


### Q06 — Derived CAD amount

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add `amount_cad` to `orders` by multiplying each `amount` by the illustrative conversion rate 1.35. Return every original order column and record alongside the new amount. Keep the original `amount` values unchanged.

**Requirements:**
- Use the fixed rate of 1.35 for every order, not a live exchange-rate lookup.
- Calculate on the distributed records and preserve null amounts as null derived values.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame containing all original order columns plus numeric `amount_cad`. Each value is the corresponding amount multiplied by 1.35, with all original records and amount values preserved.

**Solution:**


In [7]:
result = orders.withColumn('amount_cad',F.col('amount')*F.lit(1.35))
result.show(20, truncate=False)


### Q07 — Distinct statuses

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return a one-column DataFrame containing each distinct `status` value in `orders` once. Keep the column name `status` and include a single null row if any orders have a null status. Do not return other order fields.

**Requirements:**
- Do not normalize status text before identifying distinct values.
- Keep the distinct calculation distributed and retain one null entry when null statuses exist.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with only `status`, one row per distinct input status including null when present. There is no count column or required ordering.

**Solution:**


In [8]:
result = orders.select('status').distinct()
result.show(20, truncate=False)


### Q08 — Missing customer IDs

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count records in `orders` whose `customer_id` is null. Return a one-row DataFrame containing only `missing_customer_ids`. Count order rows rather than distinct orders or customers; zero is a valid count when no rows have a missing customer ID.

**Requirements:**
- Count null customer-ID occurrences, not nulls in other fields or distinct order IDs.
- Calculate the metric on the distributed orders rather than transferring all records to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A one-row PySpark DataFrame with the single column `missing_customer_ids`, reporting the number of orders with null customer IDs. Count records rather than distinct identifiers.

**Solution:**


In [9]:
result = orders.agg(F.when(F.count('*') == 0, 0).otherwise(F.sum(F.when(F.col('customer_id').isNull(),1).otherwise(0))).alias('missing_customer_ids'))
result.show(20, truncate=False)


### Q09 — Add literal source

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add a string column named `source_system` to `orders` with the value `ecommerce` on every row. Return all original records and columns together with the added source label, leaving the original order values unchanged.

**Requirements:**
- Apply the same string label regardless of order status or amount.
- Add the field in the distributed DataFrame without replacing any existing order values.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with every original order column plus string `source_system`, set to `ecommerce` on every row. All input record occurrences remain.

**Solution:**


In [10]:
result = orders.withColumn('source_system',F.lit('ecommerce'))
result.show(20, truncate=False)


### Q10 — Conditional band

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Classify each record in `orders` with a new `value_band` column. Use `high` when `amount` is at least 250 and `standard` otherwise, including null amounts. Retain every original row and column without changing the amount.

**Requirements:**
- Use exactly `high` and `standard` as the two labels.
- Classify each distributed order independently; a null amount receives `standard`, not a null label.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with all order columns plus string `value_band`. Amounts of 250 or more are `high`; all other amounts, including nulls, are `standard`. Record count is unchanged.

**Solution:**


In [11]:
result = orders.withColumn('value_band',F.when(F.col('amount')>=250,'high').otherwise('standard'))
result.show(20, truncate=False)


## 2. Filtering, Sorting & SQL Expressions


### Q11 — Multiple predicates

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return rows of `orders` whose `status` is exactly `completed` and whose `amount` is at least 200. Both conditions must be satisfied. Keep all original fields for qualifying orders and exclude rows with null amounts.

**Requirements:**
- The same order must satisfy the exact status match and the inclusive amount threshold.
- Keep matching records distributed, including duplicates, without changing original fields.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame of complete completed-order records whose non-null amount is at least 200. No other statuses or null amounts appear.

**Solution:**


In [12]:
result = orders.filter((F.col('status')=='completed') & (F.col('amount')>=200))
result.show(20, truncate=False)


### Q12 — IN predicate

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Find customer records in `customers` whose `country` equals `CA` or `US`. Return the complete matching records and preserve every original column. Compare the stored country codes exactly.

**Requirements:**
- Do not translate country names, trim codes, or ignore case.
- Retain each matching customer occurrence in the distributed result.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame containing complete customer records with country exactly `CA` or `US`, with the original customer schema.

**Solution:**


In [13]:
result = customers.filter(F.col('country').isin('CA','US'))
result.show(20, truncate=False)


### Q13 — Exclude cancelled

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return rows of `orders` whose `status` is not `cancelled`. Exclude null statuses as well as cancelled orders. Preserve all original columns for the remaining records; do not replace or normalize status values.

**Requirements:**
- Null status is not treated as an alternative status and must be excluded.
- Filter without changing status text or collecting the complete order dataset.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with the original order columns containing only non-null statuses other than `cancelled`. All qualifying record occurrences remain.

**Solution:**


In [14]:
result = orders.filter(F.col('status') != 'cancelled')
result.show(20, truncate=False)


### Q14 — Top five

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return the five highest-value records in `orders`, ranking by `amount` descending. Break equal-amount ties with the smaller `order_id` first, including at the cutoff. Preserve all original columns and return the selected records in that order.

**Requirements:**
- Apply the five-record limit after resolving amount ties by order ID.
- Rank orders in the distributed dataset; return all available orders if fewer than five exist.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame of up to five complete order records, sorted by amount descending and order ID ascending. The tie-breaker also determines which records survive the cutoff.

**Solution:**


In [15]:
result = orders.orderBy(F.desc('amount'),F.asc('order_id')).limit(5)
result.show(20, truncate=False)


### Q15 — Multi-column sort

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Sort all records in `customers` by `country` ascending, then `signup_date` ascending within each country. Place null values before non-null values for both sort keys. Retain every original record and column in the ordered result.

**Requirements:**
- Retain null-key records and place nulls first for each specified sort key.
- Sort the distributed customer records without introducing a third tie-breaker.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame containing all customer records and columns, ordered by country and then signup date ascending, with nulls first on both keys. Records tied on both keys have no additional specified order.

**Solution:**


In [16]:
result = customers.orderBy('country','signup_date')
result.show(20, truncate=False)


### Q16 — Case insensitive

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Filter `customers` to records whose `segment` equals `enterprise`, ignoring letter case. Return complete customer records with all original columns. A missing segment or a longer value merely containing that word should not qualify.

**Requirements:**
- Ignore case for comparison only, preserving the original segment values in returned records.
- Keep the filter distributed and exclude null segments and non-exact matches.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame of complete customer records whose segment equals `enterprise` without regard to case. Original values and column types remain unchanged.

**Solution:**


In [17]:
result = customers.filter(F.lower(F.col('segment'))=='enterprise')
result.show(20, truncate=False)


### Q17 — Log keyword match

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Identify `pipeline_logs` records whose `message` contains `error` or `fail` anywhere in the text, ignoring letter case. Return all original fields for qualifying logs and exclude records with null messages.

**Requirements:**
- Search `message`, not `level`, and retain a record only once even if both keywords match.
- Run the text filter on the distributed logs; null messages do not qualify.

**Available datasets and columns:**
- `pipeline_logs`: `log_id`, `pipeline`, `level`, `duration_ms`, `timestamp`, `message`

**Expected output:** A PySpark DataFrame containing complete log records with a case-insensitive `error` or `fail` substring in their message. Original message text and all other log fields are preserved.

**Solution:**


In [18]:
result = pipeline_logs.filter(F.lower(F.col('message')).rlike('error|fail'))
result.show(20, truncate=False)


### Q18 — Range predicate

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return complete rows from `transactions` whose `amount` is between 100 and 500, including both boundaries. Preserve every original transaction column. Null amounts should not qualify.

**Requirements:**
- Include amounts of exactly 100 and 500, excluding nulls.
- Filter the distributed transaction records without aggregating or deduplicating them.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** A PySpark DataFrame of complete transactions with amounts in the inclusive range 100–500. It retains every original transaction column and all qualifying occurrences.

**Solution:**


In [19]:
result = transactions.filter(F.col('amount').between(100,500))
result.show(20, truncate=False)


### Q19 — Null-safe fallback

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add `cleaned_email` to `customers`, copying the existing `email` value unless it is null, in which case use the string `unknown`. Do not replace blank but non-null strings. Preserve all rows and original columns, including the unchanged email field.

**Requirements:**
- Only null triggers the fallback; empty and whitespace-only strings remain unchanged.
- Do not trim, lowercase, or validate emails while adding the distributed derived field.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame with all original customer columns plus `cleaned_email`. The new field copies non-null email values exactly and contains `unknown` for nulls; all customers remain.

**Solution:**


In [20]:
result = customers.withColumn('cleaned_email',F.coalesce(F.col('email'),F.lit('unknown')))
result.show(20, truncate=False)


### Q20 — SQL expression

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Using the `orders` DataFrame, add a column named `gross_amount` containing each order's amount increased by 5%. Preserve all existing rows and columns, including the original `amount` values. Use PySpark SQL expressions through `expr` for the calculation.

**Requirements:**
- This is a row-level calculation: do not filter, duplicate, or aggregate orders.
- Keep the calculation distributed and use the SQL expression technique required by the Task; do not transfer the complete `orders` dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame containing every original `orders` column plus `gross_amount`. Each `gross_amount` is the corresponding `amount` increased by 5%. The output has the same number of records as `orders`, and the original `amount` values remain unchanged.

**Solution:**


In [21]:
result = orders.withColumn('gross_amount',F.expr('amount * 1.05'))
result.show(20, truncate=False)


## 3. Aggregations & GroupBy


### Q21 — Order status counts

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Create a DataFrame with one row for each distinct `status` value in `orders`, including null statuses. Return the status and the number of order rows in that group as `count`. Keep the aggregation distributed and do not require a particular output order.

**Requirements:**
- Include every order occurrence even when other fields are null.
- Keep null statuses together as a group rather than excluding them or labelling them with replacement text.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `status` and integer `count`, one row per distinct status including null when present. Counts total the input record count; row order is unspecified.

**Solution:**


In [22]:
result = orders.groupBy('status').count()
result.show(20, truncate=False)


### Q22 — Revenue by status

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Summarize `orders` by `status`, returning one row per status with its total `amount` as `revenue`. Include a null-status group when present. Ignore null amounts in a group's total, returning null revenue if that group has no non-null amounts.

**Requirements:**
- Do not convert null amounts to zero before summing.
- Aggregate across distributed records and retain a null-status group.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `status` and `revenue`, one row per observed status. Revenue sums non-null amounts and remains null for a group containing only null amounts.

**Solution:**


In [23]:
result = orders.groupBy('status').agg(F.sum('amount').alias('revenue'))
result.show(20, truncate=False)


### Q23 — Customer spend

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Calculate total order spend for each `customer_id` in `orders`, including every status. Return `customer_id` and `total_spend`, with one row per customer key and a null-key group when present. Sum non-null amounts; a group with only null amounts has null spend.

**Requirements:**
- Do not restrict spend to completed orders or require a customer-dimension match.
- Retain the null customer-key group and calculate totals without bringing the full order dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `customer_id` and `total_spend`, one row per observed customer key, including null. Totals cover all statuses; a group with no non-null amounts has null spend.

**Solution:**


In [24]:
result = orders.groupBy('customer_id').agg(F.sum('amount').alias('total_spend'))
result.show(20, truncate=False)


### Q24 — Order KPIs

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Produce order statistics by `status` from `orders`, including a null-status group when present. Return `status`, row count as `n`, and amount statistics as `avg`, `min`, and `max`. Count every order row, but exclude null amounts from the amount statistics.

**Requirements:**
- An order with a null amount still contributes to `n`.
- Calculate the amount statistics within each distributed status group, without filling missing amounts with zero.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `status`, `n`, `avg`, `min`, and `max`, one row per observed status including null. `n` counts records; the other measures use non-null amounts and are null when none exist.

**Solution:**


In [25]:
result = orders.groupBy('status').agg(F.count('*').alias('n'),F.avg('amount').alias('avg'),F.min('amount').alias('min'),F.max('amount').alias('max'))
result.show(20, truncate=False)


### Q25 — Segment distribution

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count records in `customers` for each distinct combination of `country` and `segment`. Return `country`, `segment`, and `count`, including combinations containing null grouping values. Count customer rows rather than distinct customer IDs.

**Requirements:**
- Count records rather than distinct customer identifiers.
- Preserve observed null-key combinations during the distributed aggregation; do not generate unobserved combinations.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame with `country`, `segment`, and integer `count`, one row per observed country-segment combination. Null country or segment values remain valid grouping values.

**Solution:**


In [26]:
result = customers.groupBy('country','segment').count()
result.show(20, truncate=False)


### Q26 — Completed revenue

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Calculate revenue from `orders` whose `status` is exactly `completed`. Return `customer_id` and `completed_revenue`, summing amounts separately for each customer key. Include only keys represented by qualifying orders, including a null customer key if present.

**Requirements:**
- Match `completed` exactly and omit keys with no qualifying orders.
- Sum non-null amounts in each distributed customer group rather than replacing null amounts with zero.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `customer_id` and `completed_revenue`, one row per key present among completed orders, including null keys. A qualifying group containing only null amounts has null revenue.

**Solution:**


In [27]:
result = orders.filter(F.col('status')=='completed').groupBy('customer_id').agg(F.sum('amount').alias('completed_revenue'))
result.show(20, truncate=False)


### Q27 — Product units

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Calculate total ordered quantity per product in `order_items`. Return one row per `product_id` with the summed `quantity` as `units`. Count units rather than item rows and retain a null product-key group if present.

**Requirements:**
- Use quantities, not item-record counts or prices.
- Aggregate items by product in the distributed dataset without discarding the null-key group.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`

**Expected output:** A PySpark DataFrame with `product_id` and `units`, one row per observed product key including null. `units` sums non-null quantities and is null when a group has none.

**Solution:**


In [28]:
result = order_items.groupBy('product_id').agg(F.sum('quantity').alias('units'))
result.show(20, truncate=False)


### Q28 — Line sales

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Calculate sales per product from `order_items`, valuing each item as `quantity` multiplied by its sale-time `unit_price`. Return one row per `product_id` with the sum as `sales`. The result should contain product-level totals rather than individual item records.

**Requirements:**
- Use the item's sale-time `unit_price`, not a catalog price.
- Calculate item values and product totals on the distributed records, retaining a null product-key group when present.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`

**Expected output:** A PySpark DataFrame with `product_id` and `sales`, one row per observed product key. Sales sum non-null item values; no individual item columns or intermediate line totals are returned.

**Solution:**


In [29]:
result = order_items.withColumn('line_total',F.col('quantity')*F.col('unit_price')).groupBy('product_id').agg(F.sum('line_total').alias('sales'))
result.show(20, truncate=False)


### Q29 — Conditional counts

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count completed and cancelled orders separately for every customer key represented in `orders`. Return `customer_id`, `completed`, and `cancelled`, matching those status strings exactly. Include a null customer-key group when present and report zero for either status absent from a customer's orders.

**Requirements:**
- Customers represented only by other statuses must remain with zero in both counts.
- Count order occurrences within the distributed customer groups, including the null-key group.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `customer_id`, `completed`, and `cancelled`, one row per observed customer key. The two numeric fields are status-specific record counts, with zero for an absent status.

**Solution:**


In [30]:
result = orders.groupBy('customer_id').agg(F.sum(F.when(F.col('status')=='completed',1).otherwise(0)).alias('completed'),F.sum(F.when(F.col('status')=='cancelled',1).otherwise(0)).alias('cancelled'))
result.show(20, truncate=False)


### Q30 — Approx distinct

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Estimate the number of distinct non-null `customer_id` values for each `status` in `orders` using approximate distinct counting. Return `status` and `approx_customers`, including a null-status group when present. Do not substitute the number of order rows for the number of customers.

**Requirements:**
- Use the approximate distinct-counting technique required by the Task, excluding null customer IDs.
- Repeated customer IDs within a status must not count as additional customers; keep the estimation distributed.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `status` and integer estimate `approx_customers`, one row per observed status including null. Estimates describe distinct non-null customers and need not equal exact counts.

**Solution:**


In [31]:
result = orders.groupBy('status').agg(F.approx_count_distinct('customer_id').alias('approx_customers'))
result.show(20, truncate=False)


## 4. Joins & Data Modeling


### Q31 — Enrich orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Enrich every row of `orders` with `name` and `segment` from `customers`, matching on `customer_id`. Return all original order columns and the two added fields. Keep unmatched orders with null customer fields; null keys should not match each other.

**Requirements:**
- Use ordinary key equality so null keys do not match.
- Keep the enrichment distributed and add only the requested customer fields.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame containing all original order columns plus `name` and `segment`. Matching orders carry customer values; unmatched orders, including null-key orders, remain with null added fields.

**Solution:**


In [32]:
result = orders.join(customers.select('customer_id','name','segment'),'customer_id','left')
result.show(20, truncate=False)


### Q32 — Customer order counts

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Attach an `order_count` to every record in `customers`, counting matching `orders` by `customer_id` across all statuses. Preserve all original customer columns and use zero where no order matches. Null customer keys should not match null order keys.

**Requirements:**
- Count order occurrences rather than distinct order IDs, across all statuses.
- Keep null-key customers unmatched and perform the counts and enrichment on the distributed data.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame containing all customer columns plus `order_count`, one row per input customer record. Customers without matching non-null order keys receive zero.

**Solution:**


In [33]:
counts=orders.groupBy('customer_id').count().withColumnRenamed('count','order_count');result=customers.join(counts,'customer_id','left').fillna({'order_count':0})
result.show(20, truncate=False)


### Q33 — Order line enrichment

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Enrich `order_items` with `status` and `order_date` from `orders`, matching on `order_id`. Retain every item row and all original item columns. Leave the added fields null for items without a matching order.

**Requirements:**
- Match by order ID and add only `status` and `order_date`.
- Keep unmatched and duplicate item occurrences in the distributed enrichment result.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with all original item columns plus `status` and `order_date`. Each item is enriched from its matching order; unmatched items remain with null added fields.

**Solution:**


In [34]:
result = order_items.join(orders.select('order_id','status','order_date'),'order_id','left')
result.show(20, truncate=False)


### Q34 — Product category enrichment

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add product `category` to each record in `order_items`, matching `product_id` against `products`. Preserve all original item columns and retain items without a matching product, using null for their category.

**Requirements:**
- Add category without replacing the item's sale-time unit price.
- Perform the lookup on distributed data and retain items with absent or null product matches.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`
- `products`: `product_id`, `sku`, `category`, `price`

**Expected output:** A PySpark DataFrame with all order-item columns plus `category`. Item records remain present, and those without a matching product have null category.

**Solution:**


In [35]:
result = order_items.join(products.select('product_id','category'),'product_id','left')
result.show(20, truncate=False)


### Q35 — Category revenue

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Calculate item revenue by product category using `order_items` and `products`. Match on `product_id` and value each item using its quantity and sale-time unit price. Return `category` and `revenue`, retaining unmatched items in the null-category group.

**Requirements:**
- Keep unmatched items in the category aggregation rather than dropping them.
- Use item-level sale-time values and compute the grouped totals without collecting the complete sources.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`
- `products`: `product_id`, `sku`, `category`, `price`

**Expected output:** A PySpark DataFrame with `category` and `revenue`, one row per represented category. Unmatched items and items with null category contribute to the null-category group; revenue sums item quantities multiplied by unit prices.

**Solution:**


In [36]:
result = order_items.join(products.select('product_id','category'),'product_id','left').groupBy('category').agg(F.sum(F.col('quantity')*F.col('unit_price')).alias('revenue'))
result.show(20, truncate=False)


### Q36 — Left anti join

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Use a left anti join to find customers in `customers` whose `customer_id` has no matching row in `orders`. Return all original customer columns. Orders in any status count as matches, but null keys should not match each other.

**Requirements:**
- Use the left anti join required by the Task with ordinary, not null-safe, key matching.
- Check all orders regardless of status, without collecting their keys into a driver-side lookup.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame of complete customer records with no matching order key. Null-key customers are included; no order fields are added.

**Solution:**


In [37]:
result = customers.join(orders.select('customer_id').distinct(),'customer_id','left_anti')
result.show(20, truncate=False)


### Q37 — Orphan orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return records in `orders` whose `customer_id` does not match any customer in `customers`. Treat orders with a null customer ID as orphans as well. Preserve all original order columns and do not add customer fields to the result.

**Requirements:**
- Treat null order customer IDs as unmatched even if a null key exists in `customers`.
- Keep each orphan occurrence in the distributed result rather than returning only distinct keys.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame of complete orphan orders with all original order columns. It includes null-key orders and non-null keys absent from the customer lookup.

**Solution:**


In [38]:
result = orders.join(customers.select('customer_id'),'customer_id','left_anti')
result.show(20, truncate=False)


### Q38 — Broadcast dimension

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Enrich `orders` with customer `segment`, using a broadcast join for the `customers` dimension and matching on `customer_id`. Return all original order columns and the segment. Retain unmatched orders with a null segment.

**Requirements:**
- Broadcast the customer dimension as required by the Task, not the orders dataset.
- Keep orders distributed and add only `segment`; unmatched and null-key orders remain.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame with all original order columns plus `segment`, populated from the broadcast customer lookup. Unmatched orders have null segment values.

**Solution:**


In [39]:
result = orders.join(F.broadcast(customers.select('customer_id','segment')),'customer_id','left')
result.show(20, truncate=False)


### Q39 — Semi join

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Use a semi join to return customers in `customers` who have at least one matching order in `orders` by `customer_id`. Return only the original customer columns, without multiplying customer rows for multiple orders. Null keys should not match each other.

**Requirements:**
- Use the semi-join technique required by the Task; order multiplicity must not multiply customer records.
- Match non-null keys on distributed data without copying order columns into the result.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame containing original customer records that have at least one matching order. Each qualifying input occurrence appears once, with no order fields; null keys do not qualify.

**Solution:**


In [40]:
result = customers.join(orders.select('customer_id').distinct(),'customer_id','left_semi')
result.show(20, truncate=False)


### Q40 — Join validation

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Compare the row count of `orders` before and after enrichment with customer `name` from `customers` on `customer_id`. The join must retain unmatched orders. Return a one-row DataFrame with `before` and `after`, using the actual joined row count rather than assuming it is unchanged.

**Requirements:**
- Measure the actual joined result so duplicate customer matches, if present, are reflected in `after`.
- Execute counting on distributed data; only the two scalar counts are needed to construct the report.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A one-row PySpark DataFrame with integer `before` and `after` fields. They contain the original and joined order counts respectively; equality is observed, not assumed.

**Solution:**


In [41]:
joined=orders.join(customers.select('customer_id','name'),'customer_id','left');result=spark.createDataFrame([(orders.count(),joined.count())],['before','after'])
result.show(20, truncate=False)


## 5. Window Functions & Top-N


### Q41 — Latest order

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return exactly one most recent row from `orders` for each `customer_id`. Choose the latest `order_date`, breaking equal-date ties with the largest `order_id`. Preserve the selected records' original columns and remove any temporary selection fields; the comparison order does not require a global result ordering.

**Requirements:**
- Select a complete record per customer, not independent maxima from different orders.
- Resolve selection on distributed data and omit temporary rank fields; no global output sort is required.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with the original order columns and exactly one latest order per customer key. The winning record has the greatest order date and then greatest order ID within that customer.

**Solution:**


In [42]:
w=Window.partitionBy('customer_id').orderBy(F.desc('order_date'),F.desc('order_id'));result=orders.withColumn('rn',F.row_number().over(w)).filter('rn = 1').drop('rn')
result.show(20, truncate=False)


### Q42 — Customer running spend

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add `running_spend` to every record in `orders`, totaling `amount` through the current order separately for each customer. Sequence each customer's history by `order_date` ascending and then `order_id` ascending. Preserve all original records and columns; do not collapse them into a customer-level summary.

**Requirements:**
- The cumulative range runs from the customer's first order through the current row, not through later rows sharing its date.
- Keep the calculation distributed and use order ID to sequence same-date orders.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with all original order fields plus `running_spend`, one row per input order. The value is the customer's cumulative amount in the specified history sequence; a global result sort is not required.

**Solution:**


In [43]:
w=Window.partitionBy('customer_id').orderBy('order_date','order_id').rowsBetween(Window.unboundedPreceding,Window.currentRow);result=orders.withColumn('running_spend',F.sum('amount').over(w))
result.show(20, truncate=False)


### Q43 — Previous amount

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add `previous_amount` to each record in `orders`, using the preceding order for the same customer. Sequence each customer's orders by `order_date` ascending, then `order_id` ascending. Preserve all original rows and columns and use null for a customer's first previous amount.

**Requirements:**
- Use the immediately preceding record, even if its amount is null; do not search for an earlier non-null amount.
- Evaluate customer histories on the distributed DataFrame without mixing customers.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with all original order columns plus `previous_amount`. Each customer's first order has null in the added field; other rows carry their predecessor's amount. No global result ordering is required.

**Solution:**


In [44]:
w=Window.partitionBy('customer_id').orderBy('order_date','order_id');result=orders.withColumn('previous_amount',F.lag('amount').over(w))
result.show(20, truncate=False)


### Q44 — Next order date

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add `next_order_date` to every record in `orders`, containing the date of the next order from the same customer. Sequence by `order_date` ascending and then `order_id` ascending. Retain all original rows and columns; the last order in each customer's sequence should have a null next date.

**Requirements:**
- Use the next record in the same customer's sequence, not the next distinct date.
- Keep the calculation distributed and preserve last orders with null next dates.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with every original order column plus `next_order_date`, one row per order. Same-day successors can have the same date; the final order per customer has null.

**Solution:**


In [45]:
w=Window.partitionBy('customer_id').orderBy('order_date','order_id');result=orders.withColumn('next_order_date',F.lead('order_date').over(w))
result.show(20, truncate=False)


### Q45 — Rank by amount

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Assign a dense rank to orders by `amount` descending within each `status` in `orders`. Add the result as `dense_rank`, with equal amounts sharing a rank and no gaps between ranks. Preserve all original records and columns.

**Requirements:**
- Rank within status groups on the distributed orders.
- Do not add order ID to the ranking key, since equal amounts must share the same dense rank.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame containing all order columns plus integer `dense_rank`, starting at 1 for each status's largest amount. Equal amounts share ranks and the next distinct amount receives the next rank without a gap.

**Solution:**


In [46]:
w=Window.partitionBy('status').orderBy(F.desc('amount'));result=orders.withColumn('dense_rank',F.dense_rank().over(w))
result.show(20, truncate=False)


### Q46 — Top three customers

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Calculate total spend across all order statuses for each customer using `orders` and `customers`, assigning zero to customers without matching orders. Within each segment, rank by spend descending and customer ID ascending to break ties. Return up to three customers per segment with all customer columns, `spend`, and their sequential rank `rn`.

**Requirements:**
- Apply the three-customer limit independently within each segment, retaining smaller groups in full.
- Calculate spend and segment ranking on distributed data; zero-spend customers remain eligible.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with all customer columns plus `spend` and sequential `rn`, containing up to three customers per segment. Rank starts at 1 within each segment, using spend descending and customer ID ascending, including at the cutoff; no global order is required.

**Solution:**


In [47]:
spend=orders.groupBy('customer_id').agg(F.sum('amount').alias('spend'));x=customers.join(spend,'customer_id','left').fillna({'spend':0});w=Window.partitionBy('segment').orderBy(F.desc('spend'),F.asc('customer_id'));result=x.withColumn('rn',F.row_number().over(w)).filter('rn <= 3')
result.show(20, truncate=False)


### Q47 — Customer order count

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add `customer_order_count` to each record in `orders`, showing the total number of order rows sharing its `customer_id`. Count every status and treat null customer IDs as one group. Preserve every original order row and column rather than returning one row per customer.

**Requirements:**
- Count all record occurrences, not distinct order IDs, and include every status.
- Annotate the distributed orders rather than reducing each customer to one output row.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with all original order columns plus integer `customer_order_count`. Every row for a given customer key carries that group's total count, including the null-key group.

**Solution:**


In [48]:
w=Window.partitionBy('customer_id');result=orders.withColumn('customer_order_count',F.count('*').over(w))
result.show(20, truncate=False)


### Q48 — Spend share

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** For each row in `orders`, calculate its `amount` as a percentage of the total amount for the same customer across all statuses. Add this value as `share_pct`, returning null when the customer's total is zero. Preserve all original order rows and columns.

**Requirements:**
- Use each customer's total across all statuses as the denominator, not the overall order total.
- Keep the calculation distributed and use null, not an exception or zero, for a zero-total customer.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame containing every original order column plus `share_pct`, one row per order. Shares are percentages; rows whose customer total is zero have null share values.

**Solution:**


In [49]:
w=Window.partitionBy('customer_id');result=orders.withColumn('share_pct',F.when(F.sum('amount').over(w)!=0,F.col('amount')/F.sum('amount').over(w)*100))
result.show(20, truncate=False)


### Q49 — Deduplicate CDC

**Interview scenario:** You are building a change-data-capture replication pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Deduplicate `cdc_changes` to one record per `customer_id`. Select the latest `changed_at` and, when timestamps tie, the largest `change_id`. Return all original columns of the selected rows without temporary ranking fields, retaining delete operations at this stage.

**Requirements:**
- Select the latest complete change without merging fields or filtering operations first.
- Apply the timestamp and change-ID tie-breaker on distributed data, removing temporary selection fields.

**Available datasets and columns:**
- `cdc_changes`: `change_id`, `customer_id`, `operation`, `changed_at`, `email`

**Expected output:** A PySpark DataFrame with one latest change per customer ID and all original CDC columns. Delete operations remain eligible and no ranking column is included.

**Solution:**


In [50]:
w=Window.partitionBy('customer_id').orderBy(F.desc('changed_at'),F.desc('change_id'));result=cdc_changes.withColumn('rn',F.row_number().over(w)).filter('rn = 1').drop('rn')
result.show(20, truncate=False)


### Q50 — Days since prior order

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add `days_since_previous` to `orders`, measuring calendar days between the current order date and the preceding order date for the same customer. Sequence orders by `order_date` ascending, then `order_id` ascending. Keep all original rows and columns, with null gaps for first orders.

**Requirements:**
- Calculate calendar-date gaps within each customer's ordered history.
- Retain same-day gaps as zero and first-order gaps as null; perform the calculation on distributed rows.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with all order columns plus integer `days_since_previous`, one row per order. Values describe the gap to the preceding order under the date and order-ID sequence, without requiring global output sorting.

**Solution:**


In [51]:
w=Window.partitionBy('customer_id').orderBy('order_date','order_id');result=orders.withColumn('days_since_previous',F.datediff(F.to_date('order_date'),F.to_date(F.lag('order_date').over(w))))
result.show(20, truncate=False)


## 6. Dates, Events & Streaming Concepts


### Q51 — Parse timestamps

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Parse `transaction_ts` from `transactions` into a Spark timestamp column named `ts`. Return all original transaction rows and columns alongside the parsed timestamp. Keep the original `transaction_ts` field unchanged.

**Requirements:**
- The added field must have Spark timestamp type, not a reformatted string.
- Parse on the distributed transactions and preserve the source timestamp field and all other values.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** A PySpark DataFrame with all original transaction columns plus timestamp `ts`. Every input transaction remains, and `transaction_ts` retains its original values and type.

**Solution:**


In [52]:
result = transactions.withColumn('ts',F.to_timestamp('transaction_ts'))
result.show(20, truncate=False)


### Q52 — Monthly revenue

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Summarize total `amount` in `orders` by calendar month of `order_date`, including every order status. Return `month`, represented by the timestamp at the start of the month, and `revenue`. Keep different year-month combinations separate.

**Requirements:**
- Use year and month together and include every order status.
- Compute monthly totals on the distributed data without replacing the original source dates.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with timestamp `month` and numeric `revenue`, one row per represented calendar month. `month` identifies the start of that month, and revenue sums its order amounts.

**Solution:**


In [53]:
result = orders.withColumn('month',F.date_trunc('month',F.to_timestamp('order_date'))).groupBy('month').agg(F.sum('amount').alias('revenue'))
result.show(20, truncate=False)


### Q53 — Daily transaction count

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count transaction records in `transactions` by the UTC calendar date of `transaction_ts`. Return `day` and `count`, with one row per represented date. Count records regardless of their amount, currency, or type.

**Requirements:**
- Use UTC calendar dates and count every transaction occurrence.
- Keep daily aggregation distributed without filtering currency, type, or amount.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** A PySpark DataFrame with date `day` and integer `count`, one row per represented transaction date. Counts measure transaction records, not distinct customers.

**Solution:**


In [54]:
result = transactions.withColumn('day',F.to_date('transaction_ts')).groupBy('day').count()
result.show(20, truncate=False)


### Q54 — Event daily active users

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Calculate daily active users from `events`, deriving `day` from `event_ts` in the configured UTC session. Return `day` and `dau`, the number of distinct non-null `user_id` values. Multiple events by the same user on a date should count once.

**Requirements:**
- Count each non-null user once per UTC day across all event types and sessions.
- Keep the distinct-user calculation distributed; the same user may contribute on multiple days.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** A PySpark DataFrame with date `day` and integer `dau`, one row per represented UTC day. Multiple events by a user on that date contribute one to its active-user count.

**Solution:**


In [55]:
result = events.withColumn('day',F.to_date('event_ts')).groupBy('day').agg(F.countDistinct('user_id').alias('dau'))
result.show(20, truncate=False)


### Q55 — Session duration

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Summarize `events` by `session_id` after interpreting `event_ts` as a timestamp. Return `session_id`, earliest timestamp as `start`, latest timestamp as `end`, event count as `events`, and `duration_seconds` from first to last event. A single-event session has zero duration.

**Requirements:**
- Count input event records within each session, rather than distinct users or event IDs.
- Keep the session aggregation distributed; do not collect the complete event dataset to the driver.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** A PySpark DataFrame with one row per `session_id` and columns `session_id`, `start`, `end`, `events`, and `duration_seconds`. The `start` and `end` fields are the session's earliest and latest timestamps; `events` is its record count. `duration_seconds` is the elapsed time between those boundaries, with zero for a single-event session.

**Solution:**


In [56]:
from pyspark.sql import functions as F

# Convert ISO 8601 strings to timestamps
events_fixed = events.withColumn(
    "event_ts",
    F.to_timestamp("event_ts", "yyyy-MM-dd'T'HH:mm:ssX")
)

# Calculate session boundaries
result = (
    events_fixed
    .groupBy("session_id")
    .agg(
        F.min("event_ts").alias("start"),
        F.max("event_ts").alias("end"),
        F.count("*").alias("events")
    )
    .withColumn(
        "duration_seconds",
        F.unix_timestamp("end") - F.unix_timestamp("start")
    )
)

result.show(20, truncate=False)

### Q56 — Watermark filter

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return complete records from `orders` whose `order_date` is strictly later than September 1, 2025. Compare calendar dates, exclude orders on the watermark date, and preserve all original order columns.

**Requirements:**
- Compare dates without altering the output's original `order_date` field.
- Apply the strict boundary on distributed orders; null dates do not qualify.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame of complete orders with dates later than September 1, 2025. All original order fields remain, and records on the boundary date are excluded.

**Solution:**


In [57]:
result = orders.filter(F.to_date('order_date')>F.lit('2025-09-01').cast('date'))
result.show(20, truncate=False)


### Q57 — Hour buckets

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count records in `events` in calendar-hour buckets derived from `event_ts`. Return `hour` as the timestamp at the start of each represented hour and `count`. Keep hours on different dates separate rather than combining the same hour of day.

**Requirements:**
- Use full calendar-hour timestamps rather than hour-of-day numbers.
- Count event occurrences across all types and users in each distributed hourly group.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** A PySpark DataFrame with timestamp `hour` and integer `count`, one row per represented hour. Hour values mark the start of each calendar-hour bucket, keeping different dates separate.

**Solution:**


In [58]:
result = events.groupBy(F.date_trunc('hour',F.to_timestamp('event_ts')).alias('hour')).count()
result.show(20, truncate=False)


### Q58 — Event-type distribution

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Summarize `events` by `event_type`, returning the type and its row count as `count`. Include a null-type group when present. Count all event records rather than distinct users or sessions.

**Requirements:**
- Do not deduplicate event IDs before counting.
- Aggregate by stored event type on distributed data, preserving a null-type group.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** A PySpark DataFrame with `event_type` and integer `count`, one row per observed type including null when present. Counts together account for every event record.

**Solution:**


In [59]:
result = events.groupBy('event_type').count()
result.show(20, truncate=False)


### Q59 — Pipeline daily failures

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count `pipeline_logs` records whose `level` equals `ERROR`, ignoring case, for each `pipeline` and UTC calendar date of `timestamp`. Return `pipeline`, `day`, and `count`. Include only pipeline-date groups with matching error records.

**Requirements:**
- Classify errors from `level`, not message text, and count records rather than distinct log IDs.
- Group qualifying logs by pipeline and UTC date on distributed data; do not add zero-count dates.

**Available datasets and columns:**
- `pipeline_logs`: `log_id`, `pipeline`, `level`, `duration_ms`, `timestamp`, `message`

**Expected output:** A PySpark DataFrame with `pipeline`, date `day`, and integer `count`, one row per pipeline-day containing errors. Pipelines or dates without matching errors are absent.

**Solution:**


In [60]:
result = pipeline_logs.withColumn('day',F.to_date('timestamp')).filter(F.upper('level')=='ERROR').groupBy('pipeline','day').count()
result.show(20, truncate=False)


### Q60 — Event time windows

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Using the batch `events` DataFrame, count events separately for each `event_type` within non-overlapping, one-hour tumbling windows based on `event_ts`. Return `window`, containing its start and end timestamps, `event_type`, and `count`. Each window includes its start boundary and excludes its end boundary.

**Requirements:**
- Group only by the one-hour tumbling window and `event_type`; do not introduce user, customer, or session grouping.
- Use start-inclusive, end-exclusive windows and count each event occurrence once within its type; keep the batch aggregation distributed.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** A PySpark DataFrame with `window`, `event_type`, and integer `count`, one row per observed window-type combination. `window` is a struct with timestamp fields `start` and `end`, one hour apart. No individual event or customer/session columns are returned.

**Solution:**


In [61]:
result = events.withColumn('ts',F.to_timestamp('event_ts')).groupBy(F.window('ts','1 hour'),'event_type').count()
result.show(20, truncate=False)


## 7. Data Quality, CDC & Nested JSON


### Q61 — Normalize email

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add `email_normalized` to `customers` by trimming surrounding whitespace from `email` and converting it to lowercase. Preserve null emails as null. Retain every original row and column, including the unchanged email field.

**Requirements:**
- Preserve nulls instead of turning them into replacement text.
- Normalize into the added column on distributed records; do not overwrite the original email or discard repeated addresses.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame containing all original customer columns plus `email_normalized`, with lowercase trimmed email values or null. All input customer occurrences remain.

**Solution:**


In [62]:
result = customers.withColumn('email_normalized',F.lower(F.trim('email')))
result.show(20, truncate=False)


### Q62 — Blank emails

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return complete records from `customers` whose `email` is null or becomes empty after surrounding whitespace is trimmed. Keep the original email values and every other customer field in the result. Do not replace blank values as part of this check.

**Requirements:**
- Use trimming only for the emptiness test; preserve the original email text in the result.
- Check absence or blankness on distributed customers, not full email-format validity.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame of complete customer records whose email is null or blank after trimming. The result contains original fields only, without a normalized-email helper column.

**Solution:**


In [63]:
result = customers.filter(F.col('email').isNull() | (F.trim('email')==''))
result.show(20, truncate=False)


### Q63 — Invalid transaction amounts

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Identify records in `transactions` whose `amount` is null, zero, or negative. Return the complete qualifying transaction rows with all original columns. Strictly positive amounts should be excluded.

**Requirements:**
- Retain zero amounts as invalid alongside negative and null amounts.
- Apply the distributed check across all transaction types; do not modify offending values.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** A PySpark DataFrame containing complete transaction records with null or nonpositive amounts. Original columns and qualifying duplicate occurrences are retained.

**Solution:**


In [64]:
result = transactions.filter(F.col('amount').isNull() | (F.col('amount')<=0))
result.show(20, truncate=False)


### Q64 — Duplicate customer emails

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Normalize `customers.email` by trimming surrounding whitespace and converting to lowercase. Return all customer records whose non-null normalized value occurs more than once, including every occurrence. Preserve the original customer columns and add the normalized value as `normalized`; do not return a grouped count table.

**Requirements:**
- Return every record in each repeated non-null normalized-email group, including its first occurrence.
- Exclude null normalized emails, retain original email values, and detect duplicates on distributed data.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame with all customer columns plus `normalized`, containing all records whose normalized email occurs more than once. It has no count column and no required ordering.

**Solution:**


In [65]:
x=customers.withColumn('normalized',F.lower(F.trim('email')));dups=x.groupBy('normalized').count().filter('count > 1');result=x.join(dups.select('normalized'),'normalized','inner')
result.show(20, truncate=False)


### Q65 — CDC current state

**Interview scenario:** You are building a change-data-capture replication pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Build current state from `cdc_changes` by selecting the latest `changed_at` for each customer, breaking ties with the largest `change_id`. Then exclude selected records whose operation is `DELETE` or `D`, ignoring case. Return original change columns without temporary ranks; do not substitute an earlier record for a deleted customer.

**Requirements:**
- Resolve latest-change ties before excluding deletes, so deleted customers are not resurrected.
- Perform selection on distributed CDC records and remove temporary ranking fields from the output.

**Available datasets and columns:**
- `cdc_changes`: `change_id`, `customer_id`, `operation`, `changed_at`, `email`

**Expected output:** A PySpark DataFrame with the original CDC columns and at most one current change per customer. Selected `DELETE` or `D` operations are excluded case-insensitively; no earlier replacement record is returned.

**Solution:**


In [66]:
w=Window.partitionBy('customer_id').orderBy(F.desc('changed_at'),F.desc('change_id'));result=cdc_changes.withColumn('rn',F.row_number().over(w)).filter('rn = 1').filter(~F.upper('operation').isin('DELETE','D')).drop('rn')
result.show(20, truncate=False)


### Q66 — Schema type audit

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Inspect the schema of `orders` and return a DataFrame with one row per field. Include `column` for the field name, `dtype` for its Spark simple type string, and `nullable` for whether the schema permits nulls. Report schema properties rather than inspecting individual values.

**Requirements:**
- Read field metadata rather than scanning rows to infer types or nullability.
- Schema metadata may be inspected on the driver; do not collect the underlying orders.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with string `column`, string `dtype`, and Boolean `nullable`, one row per field in the `orders` schema. `nullable` describes schema permission, not whether nulls were observed.

**Solution:**


In [67]:
result=spark.createDataFrame([(x.name,x.dataType.simpleString(),x.nullable) for x in orders.schema.fields],['column','dtype','nullable'])
result.show(20, truncate=False)


### Q67 — Order-line reconciliation

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Reconcile `orders` with `order_items` by `order_id`, calculating each item's value from quantity and sale-time unit price. Return original order columns plus `line_total` where the absolute difference from `amount` exceeds 0.01. Use zero for absent item totals during comparison, but keep the returned `line_total` null for orders without items.

**Requirements:**
- Use sale-time item prices and compare unrounded totals against the strict 0.01 tolerance.
- Compute the reconciliation on distributed data; zero-filling for comparison must not overwrite the returned null item total.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`

**Expected output:** A PySpark DataFrame with all original order columns plus `line_total`, containing only mismatches greater than 0.01. An order without items is compared against zero but retains null `line_total` if returned.

**Solution:**


In [68]:
lines=order_items.groupBy('order_id').agg(F.sum(F.col('quantity')*F.col('unit_price')).alias('line_total'));result=orders.join(lines,'order_id','left').filter(F.abs(F.col('amount')-F.coalesce(F.col('line_total'),F.lit(0)))>0.01)
result.show(20, truncate=False)


### Q68 — Missing key summary

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return a two-row DataFrame profiling missing keys in `orders` and `customers`. Use columns `dataset` and `null_key_rows`: the orders row should count null `order_id` values, and the customers row should count null `customer_id` values. Include both rows even when either count is zero.

**Requirements:**
- Inspect `order_id` for orders and `customer_id` for customers, not every nullable field.
- Count on the distributed inputs; only the two scalar totals are needed for the report.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A two-row PySpark DataFrame with `dataset` and integer `null_key_rows`. The dataset labels are `orders` and `customers`; both rows are present even when a count is zero.

**Solution:**


In [69]:
result=spark.createDataFrame([('orders',orders.filter(F.col('order_id').isNull()).count()),('customers',customers.filter(F.col('customer_id').isNull()).count())],['dataset','null_key_rows'])
result.show(20, truncate=False)


### Q69 — Parse nested JSON

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Read `events.jsonl` with an inferred schema that preserves nested `properties` as a struct. Return a DataFrame containing only `event_id`, `event_type`, and `properties` for every event. Keep the properties nested rather than flattening its fields into separate columns.

**Requirements:**
- Use the inferred nested schema required by the Task and keep `properties` as a struct.
- Read and project events as distributed records rather than loading the whole JSONL file on the driver.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** A PySpark DataFrame with only `event_id`, `event_type`, and struct `properties`, one row per event. Nested property fields remain inside the struct rather than becoming top-level columns.

**Solution:**


In [70]:
result = events.select('event_id','event_type','properties')
result.show(20, truncate=False)


### Q70 — Malformed JSON quarantine

**Interview scenario:** You are building an event-ingestion quarantine pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Read `malformed_events.jsonl` in PERMISSIVE mode, retaining corrupt input text in `_corrupt_record`. Return only malformed records with `event_id`, `event_type`, and `_corrupt_record`. Keep the corrupt text available for investigation and exclude successfully parsed records from the quarantine result.

**Requirements:**
- Use PERMISSIVE parsing and preserve `_corrupt_record` as required by the Task.
- Quarantine distributed malformed records without dropping their raw text or including successfully parsed rows.

**Available datasets and columns:**
- `malformed_events`: `event_id`, `event_type`, `_corrupt_record`

**Expected output:** A PySpark DataFrame with `event_id`, `event_type`, and `_corrupt_record`, containing only malformed input records. The corrupt-record field retains their source text for investigation.

**Solution:**


In [71]:
malformed_events.cache().count();result=malformed_events.filter(F.col('_corrupt_record').isNotNull()).select('event_id','event_type','_corrupt_record')
result.show(20, truncate=False)


## 8. Production ETL & Spark Performance


### Q71 — Idempotent union

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Create two overlapping batches from `orders`: records with `order_id` at most 100, and records with `order_id` at most 150. Combine the batches and retain one record per `order_id`, preserving all original order columns. The overlap must not produce duplicate order IDs in the final result.

**Requirements:**
- Define batches by order-ID thresholds, not physical row positions.
- Combine records on the distributed dataset and deduplicate by order ID; no winner rule for conflicting versions is specified.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with all original order columns and one record per order ID present in either batch. The shared records from the two batches appear only once.

**Solution:**


In [72]:
a=orders.filter(F.col('order_id')<=100);b=orders.filter(F.col('order_id')<=150);result=a.unionByName(b).dropDuplicates(['order_id'])
result.show(20, truncate=False)


### Q72 — Partition distribution

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Profile `customers` by counting records for each `country`, including null country values. Return `country` and `count`, sorted by count descending, so the distribution can inform a partitioning strategy. This exercise asks for the distribution report, not a repartitioned DataFrame.

**Requirements:**
- Count customer occurrences, including the null-country group, without changing physical partitions.
- Aggregate and sort the distributed summary by count only; no country tie-breaker is required.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** A PySpark DataFrame with `country` and integer `count`, one row per observed country including null. Rows are sorted by count descending, with no prescribed order among equal counts.

**Solution:**


In [73]:
result = customers.groupBy('country').count().orderBy(F.desc('count'))
result.show(20, truncate=False)


### Q73 — Repartition by key

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Repartition `orders` into eight partitions using `customer_id` as the partitioning key. Return the resulting distributed DataFrame with all original records and columns unchanged. The requested change is to data distribution, not to the logical order contents.

**Requirements:**
- Use `customer_id` for the requested eight-way repartitioning, not arbitrary row splitting.
- Retain duplicate occurrences, null values, and the full schema; keep records distributed.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with exactly eight partitions, partitioned by `customer_id`, and the same schema and complete record multiset as `orders`. No logical aggregation or row ordering is required.

**Solution:**


In [74]:
result = orders.repartition(8,'customer_id')
result.show(20, truncate=False)


### Q74 — Coalesce partitions

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Coalesce `orders` to at most two partitions without introducing a shuffle for this operation. Reduce inputs with more than two partitions to two; retain the existing count when it is already two or fewer. Preserve all records, values, and schema; no row ordering is required.

**Requirements:**
- Keep the records distributed and preserve duplicate records and null values; only the partition layout should change.
- The coalescing step must not introduce a shuffle. Existing shuffles in the upstream input plan are not prohibited.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with the same schema and complete collection of records as `orders`, including duplicate multiplicities. Its partition count is the smaller of two and the input partition count. Record order is not part of the expected result.

**Solution:**


In [75]:
result = orders.coalesce(2)
result.show(20, truncate=False)


### Q75 — Cache reused data

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Cache `orders` for reuse and trigger an action that materializes the cache. From the cached data, return complete records whose `status` is exactly `completed`. Preserve all original order columns and leave the cached input available for subsequent work.

**Requirements:**
- Materialize the cache with an action; merely declaring caching is insufficient.
- Filter the cached distributed input and leave it available for reuse rather than collecting all orders or releasing the cache.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame of complete completed-order records, plus a materialized cached version of the original orders available for later work. The result retains all original order columns.

**Solution:**


In [76]:
cached=orders.cache();cached.count();result=cached.filter(F.col('status')=='completed')
result.show(20, truncate=False)


### Q76 — Skew inspection

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Find the ten most frequent `customer_id` keys in `orders`, counting all order rows and including a null key if present. Return `customer_id` and `count`, ordered by count descending. At a tied cutoff, any tied key may be selected, but no key with a strictly greater count may be omitted.

**Requirements:**
- Count every occurrence per key, including null, before applying the ten-key limit.
- Keep frequency calculation distributed; do not invent a customer-ID tie-breaker at the cutoff.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `customer_id` and integer `count` for the ten most frequent keys, ordered by count descending. Each selected key appears once; boundary ties may vary, but all strictly higher-frequency keys must be included.

**Solution:**


In [77]:
result = orders.groupBy('customer_id').count().orderBy(F.desc('count')).limit(10)
result.show(20, truncate=False)


### Q77 — Pipeline SLA

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Summarize `pipeline_logs` by `pipeline`, returning the total number of log records as `runs` and the number whose `duration_ms` exceeds 60,000 as `sla_breaches`. Retain pipelines with no breaches and report zero for that count. A duration exactly at the threshold is not a breach.

**Requirements:**
- Count every log record in `runs`, including records with missing durations.
- Count only durations strictly above 60,000 as breaches and keep the per-pipeline aggregation distributed.

**Available datasets and columns:**
- `pipeline_logs`: `log_id`, `pipeline`, `level`, `duration_ms`, `timestamp`, `message`

**Expected output:** A PySpark DataFrame with `pipeline`, integer `runs`, and integer `sla_breaches`, one row per observed pipeline. Pipelines with no qualifying durations have zero breaches.

**Solution:**


In [78]:
result = pipeline_logs.groupBy('pipeline').agg(F.count('*').alias('runs'),F.sum(F.when(F.col('duration_ms')>60000,1).otherwise(0)).alias('sla_breaches'))
result.show(20, truncate=False)


### Q78 — Currency normalization

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add `amount_cad` to `transactions` by converting `amount` using these illustrative rates: USD 1.35, CAD 1.00, EUR 1.48, and GBP 1.72. Use the transaction's `currency` to select the rate. Set `amount_cad` to null for unrecognized currencies, and preserve all original rows and columns.

**Requirements:**
- Match currency codes exactly to the fixed illustrative rates, without fetching external rates.
- Calculate on distributed transactions and retain unsupported-currency records with null converted amounts.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** A PySpark DataFrame containing all original transaction columns plus `amount_cad`. The added field uses the currency-specific rate and is null for unrecognized currencies or null amounts; no records are removed.

**Solution:**


In [79]:
rates=F.create_map(F.lit('USD'),F.lit(1.35),F.lit('CAD'),F.lit(1.0),F.lit('EUR'),F.lit(1.48),F.lit('GBP'),F.lit(1.72));result=transactions.withColumn('amount_cad',F.col('amount')*F.element_at(rates,F.col('currency')))
result.show(20, truncate=False)


### Q79 — Data quality metrics

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Produce a one-row quality summary from `orders` with `rows`, `missing_customer_ids`, and `negative_amounts`. Count all order records, those with a null `customer_id`, and those with `amount` strictly below zero, respectively. Do not count null or zero amounts as negative.

**Requirements:**
- Count each metric independently: an order may contribute to both issue counts.
- Keep the summary distributed and exclude null and zero amounts from the negative-amount metric.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A one-row PySpark DataFrame with `rows`, `missing_customer_ids`, and `negative_amounts`. These fields report the total order count, null-customer-ID count, and strictly negative-amount count respectively.

**Solution:**


In [80]:
result=orders.agg(F.count('*').alias('rows'),F.when(F.count('*') == 0, 0).otherwise(F.sum(F.when(F.col('customer_id').isNull(),1).otherwise(0))).alias('missing_customer_ids'),F.when(F.count('*') == 0, 0).otherwise(F.sum(F.when(F.col('amount')<0,1).otherwise(0))).alias('negative_amounts'))
result.show(20, truncate=False)


### Q80 — Explain physical plan

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Build a customer-level revenue summary from `orders`, including only rows whose `status` is `completed`. Return `customer_id` and summed `amount` as `revenue`. Print the query's physical execution plan as well as the result so its distributed execution can be inspected.

**Requirements:**
- Match `completed` exactly and retain a customer-key group only when qualifying orders exist.
- Inspect the distributed query plan in addition to producing revenue data; a plan alone is not the result.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** A PySpark DataFrame with `customer_id` and `revenue`, one row per key represented by completed orders, together with a printed physical execution plan and a display of the result. Revenue sums qualifying order amounts.

**Solution:**


In [ ]:
result=orders.filter(F.col('status')=='completed').groupBy('customer_id').agg(F.sum('amount').alias('revenue'));result.explain(mode='formatted')
result.show(20, truncate=False)
